### Notebook 01 — Bigram Language Model

#### Introdução

Neste notebook construiremos nosso primeiro modelo de linguagem.

Embora extremamente simples, ele contém algumas das ideias fundamentais que aparecem em praticamente todos os modelos modernos de IA.

O objetivo não é criar um modelo poderoso.

Nosso objetivo é entender, passo a passo, como uma máquina pode aprender padrões presentes em um texto e utilizá-los para gerar novas sequências.

Ao longo desta trilha veremos uma evolução gradual dos modelos:

**Bigrama** → **Neural Bigrama** → **Redes Neurais** → **Embeddings** → **Attention** → **Transformers** → **LLMs**

Este notebook representa o primeiro passo dessa jornada.

---

#### O que vamos aprender

Ao final deste notebook você deverá entender:

- O que é um token
- O que é um vocabulário
- Como texto vira números
- O que é uma probabilidade condicional
- Como um modelo gera texto
- O que é uma Cadeia de Markov
- Por que contexto limitado é um problema

---

#### Conceito Importante

Todo modelo de linguagem, desde um simples bigrama até um GPT moderno, realiza essencialmente a mesma tarefa:

> Prever qual token provavelmente vem em seguida.

A diferença entre os modelos está na quantidade de contexto que conseguem utilizar para realizar essa previsão.

---

### 1. Carregando o Dataset

#### Objetivo

Todo modelo de linguagem precisa aprender a partir de exemplos.

Esses exemplos compõem o **dataset**, que nada mais é do que um conjunto de textos utilizados durante o treinamento.

Neste primeiro notebook utilizaremos um dataset extremamente pequeno.

O objetivo não é construir um modelo poderoso, mas compreender os conceitos fundamentais por trás dos modelos de linguagem.

Mais adiante utilizaremos datasets maiores e mais realistas.

---

#### O que é um Dataset?

Um dataset é o conjunto de informações utilizado para ensinar o modelo.

Exemplos:

- livros
- artigos
- diálogos
- código-fonte
- páginas da internet

Modelos modernos são treinados com bilhões ou até trilhões de tokens.

Nosso primeiro modelo será treinado com apenas algumas linhas de texto.

In [3]:
# ==================================================
# SEÇÃO 1 - CARREGAR DATASET
# ==================================================

with open("../../data/input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text)

ola mundo
o gato dormiu
o cachorro latiu
ola chatgpt


#### Exercício de Reflexão

Observe o conteúdo carregado.

Pergunta:

O modelo está enxergando palavras, frases ou apenas texto bruto?

Pense na resposta antes de continuar.

#### Conceito Importante

Neste momento ainda não existe tokenização.

O modelo possui apenas uma sequência de caracteres.

Para o computador, texto é apenas uma sequência de símbolos armazenados na memória.

---

### 2. Criando o Vocabulário

#### Objetivo

Descobrir todos os símbolos únicos presentes no dataset.

Esse conjunto de símbolos é chamado de **vocabulário**.

Todo modelo de linguagem possui algum tipo de vocabulário.

Sem ele, o modelo não consegue representar texto matematicamente.

---

#### O que é um Vocabulário?

Vocabulário é o conjunto de tokens que o modelo conhece.

Neste notebook cada caractere será tratado como um token.

Exemplo:

Texto:

```
ola
```

Vocabulário:

```python
['o', 'l', 'a']
```

Quantidade de tokens:

```
3
```

In [4]:
# ==================================================
# SEÇÃO 2 - VOCABULÁRIO
# ==================================================

chars = sorted(list(set(text)))

vocab_size = len(chars)

print(chars)
print()
print(f"Vocabulary Size: {vocab_size}")

['\n', ' ', 'a', 'c', 'd', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 't', 'u']

Vocabulary Size: 16


---

### 3. Tokenização

#### Objetivo

Converter texto em números.

Modelos de linguagem não trabalham diretamente com texto.

Toda informação precisa ser representada numericamente antes de ser processada.

Esse processo é chamado de **tokenização**.

---

#### Por que converter texto para números?

Operações matemáticas são realizadas sobre números.

Como modelos computacionais trabalham manipulando valores numéricos, precisamos transformar cada símbolo do texto em um identificador.

---

#### Exemplo

Texto:

```
ola
```

Vocabulário:

```python
['a', 'l', 'o']
```

Mapeamento:

```python
a → 0
l → 1
o → 2
```

Texto tokenizado:

```python
[2, 1, 0]
```

---

#### Encode e Decode

Vamos criar duas funções:

- `encode` → texto para números
- `decode` → números para texto

Essas funções devem ser inversas uma da outra.

Ou seja:

```python
decode(encode(texto))
```

deve sempre reconstruir o texto original.

In [5]:
# ==================================================
# SEÇÃO 3 - TOKENIZAÇÃO
# ==================================================

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join([itos[i] for i in ids])

print("Encode:")
print(encode("ola"))

print()

print("Decode:")
print(decode(encode("ola")))

Encode:
[11, 8, 2]

Decode:
ola


#### Exercício de Reflexão

Considere o texto:

```
gato
```

Pergunta:

O modelo está armazenando a palavra "gato" ou apenas os IDs correspondentes aos caracteres?

O que acontece com o significado da palavra durante essa transformação?

#### Conceito Importante

Tokenização não significa compreensão.

O modelo apenas substitui símbolos por números.

Neste momento não existe significado.

Existe apenas uma representação numérica do texto.

O significado será aprendido posteriormente através dos padrões observados durante o treinamento.

---

### 4. Transformando o Dataset em Tokens

#### Objetivo

Aplicar o processo de tokenização ao dataset inteiro.

A partir deste momento o modelo passará a trabalhar apenas com números.

Esse é exatamente o mesmo princípio utilizado em modelos modernos, embora os tokens sejam muito mais sofisticados.

---

#### O que muda agora?

Até aqui possuíamos:

```text
ola mundo
o gato dormiu
o cachorro latiu
```

Após a tokenização teremos algo parecido com:

```python
[12, 9, 3, 1, 15, 14, 8, 7, 13, ...]
```

Os valores exatos dependem do vocabulário criado.

---

#### Por que isso é importante?

Porque modelos de linguagem realizam operações matemáticas.

Eles não enxergam letras, palavras ou frases.

Eles enxergam apenas sequências numéricas.

In [6]:
# ==================================================
# SEÇÃO 4 - DATASET TOKENIZADO
# ==================================================

data = encode(text)

print(data[:100])

[11, 8, 2, 1, 9, 15, 10, 4, 11, 0, 11, 1, 5, 2, 14, 11, 1, 4, 11, 13, 9, 7, 15, 0, 11, 1, 3, 2, 3, 6, 11, 13, 13, 11, 1, 8, 2, 14, 7, 15, 0, 11, 8, 2, 1, 3, 6, 2, 14, 5, 12, 14]


#### Exercício de Reflexão

Observe a saída gerada.

Você ainda consegue identificar facilmente o texto original?

Por que uma sequência numérica é mais útil para um computador do que texto puro?

#### Conceito Importante

A partir daqui o modelo não enxerga mais palavras nem frases.

Ele vê apenas uma sequência de IDs.

Todo o processamento realizado por modelos de linguagem acontece sobre representações numéricas como essa.

---

### 5. Construindo os Bigramas

#### Objetivo

Agora vamos começar a extrair conhecimento do dataset.

Até este ponto nosso modelo apenas armazenava texto em formato numérico.

A partir desta seção ele começará a observar relações entre tokens consecutivos.

Essas relações são chamadas de **bigramas**.

---

#### O que é um Bigrama?

Um bigrama é um par formado por dois tokens consecutivos.

Exemplo:

Texto:

```
ola
```

Tokens:

```
o → l → a
```

Bigramas:

```
(o, l)
(l, a)
```

---

#### O que o modelo está tentando descobrir?

A pergunta central é:

> Dado o token atual, qual token costuma aparecer em seguida?

Essa é uma das ideias fundamentais dos modelos de linguagem.

Mesmo modelos modernos continuam tentando prever o próximo token.

A diferença está na quantidade de contexto utilizada para fazer essa previsão.

In [7]:
# ==================================================
# SEÇÃO 5 - CONSTRUÇÃO DOS BIGRAMAS
# ==================================================

bigrams = {}

for i in range(len(data) - 1):

    current_token = data[i]
    next_token = data[i + 1]

    if current_token not in bigrams:
        bigrams[current_token] = {}

    if next_token not in bigrams[current_token]:
        bigrams[current_token][next_token] = 0

    bigrams[current_token][next_token] += 1

In [8]:
# ==================================================
# VISUALIZANDO TRANSIÇÕES
# ==================================================

for current_token in bigrams:

    current_char = itos[current_token]

    print(f"\nDepois de '{current_char}'")

    for next_token, count in bigrams[current_token].items():

        next_char = itos[next_token]

        print(
            f"  -> '{next_char}' : {count}"
        )


Depois de 'o'
  -> 'l' : 2
  -> '
' : 1
  -> ' ' : 4
  -> 'r' : 2

Depois de 'l'
  -> 'a' : 3

Depois de 'a'
  -> ' ' : 2
  -> 't' : 3
  -> 'c' : 1

Depois de ' '
  -> 'm' : 1
  -> 'g' : 1
  -> 'd' : 1
  -> 'c' : 2
  -> 'l' : 1

Depois de 'm'
  -> 'u' : 1
  -> 'i' : 1

Depois de 'u'
  -> 'n' : 1
  -> '
' : 2

Depois de 'n'
  -> 'd' : 1

Depois de 'd'
  -> 'o' : 2

Depois de '
'
  -> 'o' : 3

Depois de 'g'
  -> 'a' : 1
  -> 'p' : 1

Depois de 't'
  -> 'o' : 1
  -> 'i' : 1
  -> 'g' : 1

Depois de 'r'
  -> 'm' : 1
  -> 'r' : 1
  -> 'o' : 1

Depois de 'i'
  -> 'u' : 2

Depois de 'c'
  -> 'a' : 1
  -> 'h' : 2

Depois de 'h'
  -> 'o' : 1
  -> 'a' : 1

Depois de 'p'
  -> 't' : 1


#### Exercício de Reflexão

Observe as transições encontradas.

Pergunta:

Se o caractere "o" aparece várias vezes no dataset, ele sempre será seguido pelo mesmo caractere?

O que isso nos diz sobre a natureza da linguagem?

#### Conceito Importante

Neste momento o modelo ainda não entende significado.

Ele não sabe o que é:

- cachorro
- gato
- dormir
- correr

Ele apenas observa padrões estatísticos.

Seu conhecimento é composto exclusivamente pelas frequências observadas durante o treinamento.

---

### 6. Transformando Contagens em Probabilidades

#### Objetivo

Até agora nosso modelo aprendeu apenas frequências.

Por exemplo:

Depois do token `"o"` podemos ter observado:

```text
l → 5 vezes
' ' → 3 vezes
```

Mas para fazer previsões precisamos transformar essas contagens em probabilidades.

---

#### Por que probabilidades?

Imagine que queremos prever o próximo token após `"o"`.

Se utilizarmos apenas as contagens:

```text
l → 5
' ' → 3
```

Ainda não sabemos qual a chance real de cada opção.

Precisamos normalizar esses valores.

---

#### Intuição

Se um evento ocorreu:

```text
5 vezes em um total de 8
```

Então sua probabilidade é:

$$
\frac{5}{8}
$$

Ou:

$$
62.5\%
$$

É exatamente isso que faremos nesta seção.

---

#### Normalização

Para cada token atual:

1. Somamos todas as transições observadas
2. Dividimos cada contagem pelo total
3. Obtemos uma distribuição de probabilidades

A soma das probabilidades de uma linha deve ser sempre:

$$
1.0
$$

In [9]:
# ==================================================
# SEÇÃO 6 - PROBABILIDADES
# ==================================================

probabilities = {}

for current_token, transitions in bigrams.items():

    total = sum(transitions.values())

    probabilities[current_token] = {}

    for next_token, count in transitions.items():

        probabilities[current_token][next_token] = (
            count / total
        )

In [10]:
# ==================================================
# VISUALIZANDO PROBABILIDADES
# ==================================================

for current_token in probabilities:

    current_char = itos[current_token]

    print(f"\nDepois de '{current_char}'")

    for next_token, prob in probabilities[current_token].items():

        next_char = itos[next_token]

        print(
            f"  -> '{next_char}' : {prob:.3f}"
        )


Depois de 'o'
  -> 'l' : 0.222
  -> '
' : 0.111
  -> ' ' : 0.444
  -> 'r' : 0.222

Depois de 'l'
  -> 'a' : 1.000

Depois de 'a'
  -> ' ' : 0.333
  -> 't' : 0.500
  -> 'c' : 0.167

Depois de ' '
  -> 'm' : 0.167
  -> 'g' : 0.167
  -> 'd' : 0.167
  -> 'c' : 0.333
  -> 'l' : 0.167

Depois de 'm'
  -> 'u' : 0.500
  -> 'i' : 0.500

Depois de 'u'
  -> 'n' : 0.333
  -> '
' : 0.667

Depois de 'n'
  -> 'd' : 1.000

Depois de 'd'
  -> 'o' : 1.000

Depois de '
'
  -> 'o' : 1.000

Depois de 'g'
  -> 'a' : 0.500
  -> 'p' : 0.500

Depois de 't'
  -> 'o' : 0.333
  -> 'i' : 0.333
  -> 'g' : 0.333

Depois de 'r'
  -> 'm' : 0.333
  -> 'r' : 0.333
  -> 'o' : 0.333

Depois de 'i'
  -> 'u' : 1.000

Depois de 'c'
  -> 'a' : 0.333
  -> 'h' : 0.667

Depois de 'h'
  -> 'o' : 0.500
  -> 'a' : 0.500

Depois de 'p'
  -> 't' : 1.000


#### Exercício de Reflexão

Observe as probabilidades calculadas.

Pergunta:

Se após o token `"o"` encontramos:

```text
l → 0.75
' ' → 0.25
```

Isso significa que o modelo sempre escolherá `"l"`?

Ou apenas que `"l"` é mais provável?

Qual seria a vantagem de permitir escolhas diferentes ocasionalmente?

#### Conceito Importante

Probabilidade não é certeza.

Uma probabilidade de 80% não significa que um evento acontecerá sempre.

Significa apenas que ele tende a acontecer com maior frequência.

Essa distinção é extremamente importante para entender modelos de linguagem.

Modelos modernos também trabalham com distribuições de probabilidade.

---

### 7. A Matemática por Trás do Bigrama

#### Objetivo

Formalizar matematicamente o que acabamos de construir.

Nosso modelo responde sempre à mesma pergunta:

> Dado o token atual, qual é a probabilidade do próximo token?

Essa relação é chamada de probabilidade condicional.

---

#### Fórmula

$$
P(x_t \mid x_{t-1})
$$

Onde:

- $x_t$ representa o token atual
- $x_{t-1}$ representa o token anterior

Lemos essa expressão como:

> Probabilidade do token atual dado o token anterior.

---

#### Exemplo

Se observarmos a sequência:

```text
o → l → a
```

O modelo pode aprender:

$$
P(l \mid o) = 0.75
$$

e

$$
P(\text{espaço} \mid o) = 0.25
$$

Isso significa que, após observar o token `"o"`, existe 75% de chance de o próximo token ser `"l"`.

---

#### O que o modelo realmente aprendeu?

Todo o conhecimento do nosso modelo pode ser resumido em várias probabilidades condicionais:

$$
P(\text{próximo token} \mid \text{token atual})
$$

Nada mais.

Nada menos.

#### Exercício de Reflexão

Considere as frases:

```text
O cachorro latiu.
O cachorro correu.
```

Após a palavra "cachorro" existem múltiplas possibilidades.

Como um modelo baseado em bigramas decide qual delas escolher?

O que acontece quando existem várias opções válidas?

#### Conceito Importante

A principal limitação do modelo de bigramas é sua memória extremamente curta.

Ele só consegue enxergar um token anterior.

Todo o restante do contexto é descartado.

Essa característica faz com que o modelo seja simples de entender, mas também limita severamente sua capacidade de representar linguagem.

---

### 8. Gerando Texto

#### Objetivo

Até agora nosso modelo aprendeu padrões estatísticos presentes no dataset.

Ele sabe:

- quais tokens costumam aparecer depois de outros
- com que frequência essas transições ocorrem
- qual a probabilidade de cada próxima escolha

Agora vamos utilizar esse conhecimento para gerar texto.

---

#### Como a geração funciona?

O processo é simples:

1. Escolhemos um token inicial
2. Consultamos suas probabilidades
3. Sorteamos o próximo token
4. O token escolhido torna-se o novo estado atual
5. Repetimos o processo

---

#### O que significa autoregressivo?

Significa que cada nova previsão depende das previsões anteriores.

O modelo gera um token por vez.

Depois utiliza esse token para gerar o próximo.

Esse mesmo princípio continua presente nos LLMs modernos.

In [14]:
# ==================================================
# SEÇÃO 8 - GERAÇÃO DE TEXTO
# ==================================================

import random

start_char = text[0]

current_token = stoi[start_char]

generated = start_char

for _ in range(200):

    if current_token not in probabilities:
        break

    next_tokens = list(
        probabilities[current_token].keys()
    )

    probs = list(
        probabilities[current_token].values()
    )

    next_token = random.choices(
        next_tokens,
        weights=probs,
        k=1
    )[0]

    generated += itos[next_token]

    current_token = next_token

print(generated)

orrrmiu
o miu
o mu
o la chatiu
o
o ga mundoro chatiu
olacha gpto
orrmiu
o gptiundo laca cholatiu
oro cha la ga gatgpto do gatiundo
o
o
ola miu
olatgptiu
o miundoro ca ga chachorrorrrmundo gptgptiu
o ga


#### Exercício de Reflexão

Execute a célula várias vezes.

Perguntas:

- O resultado é sempre igual?
- Quais partes tendem a se repetir?
- Quais partes mudam?

Por que isso acontece?

Observe que o modelo utiliza probabilidades, não regras fixas.

#### Conceito Importante

O modelo não copia frases inteiras do dataset.

Ele gera texto realizando uma sequência de escolhas probabilísticas.

Cada nova escolha influencia todas as escolhas futuras.

Por isso pequenas diferenças podem produzir resultados completamente diferentes.

---

### 9. O Nascimento de um Modelo de Linguagem

Parabéns.

Você acabou de construir um modelo de linguagem funcional.

Ele é extremamente simples, mas já possui vários elementos fundamentais presentes em sistemas modernos:

- Tokenização
- Vocabulário
- Representação numérica
- Probabilidades
- Predição do próximo token
- Geração autoregressiva

---

#### O que o modelo aprendeu?

Durante o treinamento, o modelo observou o dataset e respondeu repetidamente à seguinte pergunta:

> Dado o token atual, qual token costuma aparecer em seguida?

O resultado desse aprendizado foi uma coleção de probabilidades condicionais.

Por exemplo:

$$
P(l \mid o)
$$

ou

$$
P(a \mid l)
$$

Essas probabilidades representam todo o conhecimento armazenado pelo modelo.

---

#### O que ele ainda não sabe fazer?

Apesar de funcional, nosso modelo possui uma limitação muito importante.

Ele enxerga apenas um token anterior.

Em termos matemáticos:

$$
P(x_t \mid x_{t-1})
$$

Todo o restante do contexto é descartado.

---

#### Um Exemplo do Problema

Considere as frases:

```text
o cachorro perseguiu o gato
```

e

```text
o gato perseguiu o cachorro
```

Nosso modelo não possui memória suficiente para compreender a estrutura completa da frase.

Ele observa apenas o token imediatamente anterior.

Isso faz com que muitas informações importantes sejam perdidas.

---

#### Uma Forma de Enxergar o Bigrama

Podemos imaginar o modelo como uma tabela gigante.

Cada linha representa um token atual.

Cada coluna representa um possível próximo token.

O modelo consulta essa tabela para decidir qual será sua próxima escolha.

Essa ideia é simples, funciona bem para exemplos pequenos e nos ajuda a entender a base dos modelos de linguagem.

Mas ela possui limitações importantes.

---

#### A Próxima Pergunta

Até aqui calculamos todas as probabilidades diretamente a partir das contagens observadas no dataset.

Mas e se, em vez de armazenar uma tabela de frequências, utilizássemos uma rede neural para aprender essas probabilidades?

Essa será exatamente a ideia do próximo notebook.

Manteremos o mesmo problema:

> prever o próximo token

Mas substituiremos as contagens por parâmetros aprendidos através de treinamento.

Essa mudança parece pequena.

Na prática, ela é o primeiro passo em direção aos modelos modernos.

---

### Resumo

Neste notebook aprendemos:

✅ O que é um token

✅ O que é um vocabulário

✅ Como texto vira números

✅ Como construir bigramas

✅ Como calcular probabilidades

✅ O que é uma probabilidade condicional

✅ Como gerar texto autoregressivamente

✅ O que é uma Cadeia de Markov de primeira ordem

✅ Por que contexto limitado é um problema

---

#### Desafio Opcional

Experimente:

- Adicionar novas frases ao dataset
- Observar como as probabilidades mudam
- Gerar textos maiores
- Alterar o caractere inicial utilizado na geração
- Criar um dataset com temas diferentes

Observe como pequenas mudanças nos dados alteram completamente o comportamento do modelo.

---

Próximo notebook:

**02 — Neural Bigrama**